# FastAPI Testing

Covers `TestClient` usage, async testing with `httpx.AsyncClient`, `dependency_overrides` for mocking, validation error testing, and auth endpoint testing.

In [ ]:
# !pip install fastapi httpx pytest anyio

from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

# --- App under test ---
app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    in_stock: bool = True

items_db: dict[int, Item] = {}
next_id = 1

@app.post("/items", status_code=201)
def create_item(item: Item):
    global next_id
    items_db[next_id] = item
    result = {"id": next_id, **item.model_dump()}
    next_id += 1
    return result

@app.get("/items/{item_id}")
def get_item(item_id: int):
    item = items_db.get(item_id)
    if not item:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"id": item_id, **item.model_dump()}

@app.get("/items")
def list_items():
    return [{"id": k, **v.model_dump()} for k, v in items_db.items()]

# --- TestClient usage ---
client = TestClient(app)

# Create
r = client.post("/items", json={"name": "Widget", "price": 9.99})
assert r.status_code == 201
print("Created:", r.json())

# Read
r = client.get("/items/1")
assert r.status_code == 200
print("Read:", r.json())

# Not found
r = client.get("/items/999")
assert r.status_code == 404
print("Not found:", r.json())

print("All basic TestClient tests passed.")

In [ ]:
# --- Async testing with httpx.AsyncClient ---
import httpx
import asyncio
from fastapi import FastAPI

app_async = FastAPI()

@app_async.get("/async-hello")
async def async_hello(name: str = "World"):
    return {"message": f"Hello, {name}!"}

@app_async.get("/async-items")
async def async_items():
    return [{"id": i, "name": f"item_{i}"} for i in range(1, 4)]

async def run_async_tests():
    async with httpx.AsyncClient(app=app_async, base_url="http://test") as ac:
        # Test 1: default name
        r = await ac.get("/async-hello")
        assert r.status_code == 200
        assert r.json() == {"message": "Hello, World!"}
        print("Async test 1:", r.json())

        # Test 2: custom name
        r = await ac.get("/async-hello?name=Alice")
        assert r.json()["message"] == "Hello, Alice!"
        print("Async test 2:", r.json())

        # Test 3: list items
        r = await ac.get("/async-items")
        assert len(r.json()) == 3
        print("Async test 3:", r.json())

    print("All async tests passed.")

await run_async_tests()

In [ ]:
# --- dependency_overrides for mocking ---
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app3 = FastAPI()

# Real dependency (would hit a real DB in production)
def get_db():
    return {"type": "real_database", "users": {}}

def get_current_user(db: dict = Depends(get_db)):
    # In real code: query db for user from token
    return db.get("current_user", {"id": 0, "name": "anonymous"})

@app3.get("/profile")
def profile(user: dict = Depends(get_current_user)):
    return user

@app3.get("/db-info")
def db_info(db: dict = Depends(get_db)):
    return {"db_type": db["type"]}

client3 = TestClient(app3)

# Without override — uses real dependency
r = client3.get("/profile")
print("Real dep:", r.json())

# Override get_current_user with a mock
def mock_current_user():
    return {"id": 42, "name": "MockUser", "role": "tester"}

def mock_db():
    return {"type": "mock_database", "users": {42: "MockUser"}}

app3.dependency_overrides[get_current_user] = mock_current_user
app3.dependency_overrides[get_db] = mock_db

r = client3.get("/profile")
print("Mocked user:", r.json())

r = client3.get("/db-info")
print("Mocked DB:", r.json())

# Clear overrides
app3.dependency_overrides.clear()
r = client3.get("/db-info")
print("After clear:", r.json())

In [ ]:
# --- Testing validation errors ---
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, field_validator

app4 = FastAPI()

class RegisterRequest(BaseModel):
    username: str
    email: str
    age: int

    @field_validator("age")
    @classmethod
    def age_must_be_adult(cls, v):
        if v < 18:
            raise ValueError("Must be 18 or older")
        return v

@app4.post("/register")
def register(req: RegisterRequest):
    return {"registered": req.username}

client4 = TestClient(app4)

# Valid
r = client4.post("/register", json={"username": "alice", "email": "a@b.com", "age": 25})
assert r.status_code == 200
print("Valid registration:", r.json())

# Missing fields
r = client4.post("/register", json={"username": "bob"})
assert r.status_code == 422
errors = r.json()["detail"]
missing_fields = [e["loc"][-1] for e in errors if e["type"] == "missing"]
print("Missing fields:", missing_fields)

# Under-age
r = client4.post("/register", json={"username": "kid", "email": "k@b.com", "age": 15})
assert r.status_code == 422
print("Under-age error:", r.json()["detail"][0]["msg"])

# Wrong type
r = client4.post("/register", json={"username": "x", "email": "x@b.com", "age": "old"})
assert r.status_code == 422
print("Wrong type error:", r.json()["detail"][0]["type"])

print("All validation tests passed.")

In [ ]:
# --- Testing auth endpoints ---
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from jose import jwt
from datetime import datetime, timedelta, timezone

SECRET = "test-secret"
ALGO = "HS256"

app5 = FastAPI()
oauth2 = OAuth2PasswordBearer(tokenUrl="/login")

USERS = {"testuser": "testpass"}

@app5.post("/login")
def login(form: OAuth2PasswordRequestForm = Depends()):
    if USERS.get(form.username) != form.password:
        raise HTTPException(status_code=401, detail="Bad credentials")
    token = jwt.encode(
        {"sub": form.username, "exp": datetime.now(timezone.utc) + timedelta(minutes=30)},
        SECRET, algorithm=ALGO
    )
    return {"access_token": token, "token_type": "bearer"}

@app5.get("/protected")
def protected(token: str = Depends(oauth2)):
    try:
        payload = jwt.decode(token, SECRET, algorithms=[ALGO])
        return {"user": payload["sub"], "status": "authenticated"}
    except Exception:
        raise HTTPException(status_code=401, detail="Invalid token")

client5 = TestClient(app5)

# Test: successful login
r = client5.post("/login", data={"username": "testuser", "password": "testpass"})
assert r.status_code == 200
token = r.json()["access_token"]
print("Login OK, token received:", bool(token))

# Test: access protected route
r = client5.get("/protected", headers={"Authorization": f"Bearer {token}"})
assert r.status_code == 200
print("Protected route:", r.json())

# Test: wrong password
r = client5.post("/login", data={"username": "testuser", "password": "wrong"})
assert r.status_code == 401
print("Wrong password:", r.json())

# Test: no token
r = client5.get("/protected")
assert r.status_code == 401
print("No token:", r.status_code)

# Test: tampered token
r = client5.get("/protected", headers={"Authorization": "Bearer fake.token.here"})
assert r.status_code == 401
print("Tampered token:", r.json())

print("All auth tests passed.")